# Transformer Decoder

## Code

In [11]:
import torch
import torch.nn as nn

import models.deep_learning.architectures as mynn


## Testing

In [12]:
# input parameters
N = 3
M = 4
batch_size = 2
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
dtype = torch.float32

# Decoder Layer parameters
d_model = 4
nhead = 2
dim_feedforward = 64
dropout = 0.2
layer_norm_eps = 1e-5
batch_first = True
norm_first = True
bias = True

## Transformer decoder parameters
num_layers = 3
#  It helps only when norm_first is True,
norm = None  # nn.LayerNorm(d_model).to(device=device,dtype=dtype)

In [13]:
torch.manual_seed(0)
x = torch.randn(batch_size, N, d_model, device=device, dtype=dtype)
memory = torch.randn(batch_size, M, d_model, device=device, dtype=dtype)

In [14]:
init_seed = 42  # avoide weights initialization randomness effects
train_seed = 24  # avoid dropout randomness effects

In [15]:
torch.manual_seed(init_seed)
tf_decl = mynn.TransformerDecoderLayer(
    d_model,
    nhead,
    dim_feedforward=dim_feedforward,
    dropout=dropout,
    activation_cls=nn.GELU,
    layer_norm_eps=layer_norm_eps,
    norm_first=norm_first,
    bias=bias,
    device=device,
    dtype=dtype,
)

torch.manual_seed(init_seed)
nn_tf_decl = nn.TransformerDecoderLayer(
    d_model,
    nhead,
    dim_feedforward=dim_feedforward,
    dropout=dropout,
    activation="gelu",
    layer_norm_eps=layer_norm_eps,
    batch_first=batch_first,
    norm_first=norm_first,
    bias=bias,
    device=device,
    dtype=dtype,
)


tf_decl.load_weights_from_torch_decoder_layer(nn_tf_decl)
tf_dec = mynn.TransformerDecoder(tf_decl, num_layers, norm=norm)
nn_tf_dec = nn.TransformerDecoder(
    nn_tf_decl,
    num_layers,
    norm=norm,
)


### Evaluation

In [16]:
tf_dec.eval()
tf_dec(x, memory)

tensor([[[-0.3635, -0.6357,  1.1700, -0.6238],
         [ 0.5206,  0.4281,  1.2181,  0.9481],
         [-0.2662,  0.1485,  0.0179, -0.3443]],

        [[-0.6304,  1.2956, -0.2041,  1.9955],
         [ 0.0226,  0.8401, -0.6081,  1.3708],
         [ 2.0342,  0.8671,  1.0781,  0.6432]]], device='mps:0',
       grad_fn=<AddBackward0>)

In [17]:
nn_tf_dec.eval()
nn_tf_dec(x, memory)

tensor([[[-0.3635, -0.6357,  1.1700, -0.6238],
         [ 0.5206,  0.4281,  1.2181,  0.9481],
         [-0.2662,  0.1485,  0.0179, -0.3443]],

        [[-0.6304,  1.2956, -0.2041,  1.9955],
         [ 0.0226,  0.8401, -0.6081,  1.3708],
         [ 2.0342,  0.8671,  1.0781,  0.6432]]], device='mps:0',
       grad_fn=<AddBackward0>)

### Training

In [18]:
mse = torch.nn.MSELoss()

In [19]:
torch.manual_seed(train_seed)
nn_tf_dec.train()
print(nn_tf_dec(x, memory))
loss = mse(nn_tf_dec(x, memory), x)
print(loss.item())
loss.backward()
nn_tf_dec(x, memory)


tensor([[[-0.3301, -1.0148,  1.4576, -0.9535],
         [ 0.7272,  0.6500,  0.9157,  0.6363],
         [-0.0927,  0.5408, -0.1163,  0.7989]],

        [[-0.8316,  2.2355,  0.4027,  2.1228],
         [-0.0283,  0.8226,  0.1561,  1.8997],
         [ 1.5845,  0.8053,  1.0972,  0.1679]]], device='mps:0',
       grad_fn=<AddBackward0>)
0.6959366202354431


tensor([[[-0.9932, -0.7123, -0.1723, -1.2139],
         [ 0.3658,  0.2793,  1.2378,  0.7991],
         [-0.0976,  0.0691,  0.3604, -0.2629]],

        [[-0.7109,  0.6426,  0.0706,  1.2728],
         [ 0.0144,  0.4316, -0.7404,  0.6500],
         [ 2.5232,  1.8907, -0.1156,  1.4840]]], device='mps:0',
       grad_fn=<AddBackward0>)

In [20]:
torch.manual_seed(train_seed)
tf_dec.train()
print(tf_dec(x, memory))
loss = mse(tf_dec(x, memory), x)
print(loss.item())
loss.backward()
tf_dec(x, memory)

tensor([[[-0.3301, -1.0148,  1.4576, -0.9535],
         [ 0.7272,  0.6500,  0.9157,  0.6363],
         [-0.0927,  0.5408, -0.1163,  0.7989]],

        [[-0.8316,  2.2355,  0.4027,  2.1228],
         [-0.0283,  0.8226,  0.1561,  1.8997],
         [ 1.5845,  0.8053,  1.0972,  0.1679]]], device='mps:0',
       grad_fn=<AddBackward0>)
0.6959365606307983


tensor([[[-0.9932, -0.7123, -0.1723, -1.2139],
         [ 0.3658,  0.2793,  1.2378,  0.7991],
         [-0.0976,  0.0691,  0.3604, -0.2629]],

        [[-0.7109,  0.6426,  0.0706,  1.2728],
         [ 0.0144,  0.4316, -0.7404,  0.6500],
         [ 2.5232,  1.8907, -0.1156,  1.4840]]], device='mps:0',
       grad_fn=<AddBackward0>)